# Stone-Level Detection Evaluation (Multi-Model)

This notebook evaluates segmentation performance at the **individual stone level** rather than pixel level, comparing **all three model configurations** (3-channel, 4-channel, 7-channel) in a single run.

**Key insight**: Individual stones in the ground truth are separated by black (background) pixels. Using connected component analysis, we can identify each stone as a distinct object and evaluate:
- **Detection Rate**: How many GT stones were detected (regardless of class)?
- **Classification Accuracy**: Of detected stones, how many have the correct class?
- **Per-Class Detection**: Detection rates broken down by masonry type

A stone is considered **correctly detected** if its predicted pixels overlap with the GT stone by a configurable threshold (default: 50% IoU).

## Workflow:
1. **Set parameters** (paths for GT and all 3 predictions, detection threshold)
2. **Extract individual stones** from GT using connected components
3. **Match stones** to predictions and calculate detection metrics
4. **Compare results** across models

---
## Cell 1: Parameters

Set your paths and detection thresholds here. Re-run subsequent cells after changing these.

In [ ]:
# ============================================================
# PARAMETERS - EDIT THESE
# ============================================================

# Ground truth mask path
GT_MASK_PATH = "/path/to/ground_truth_mask.png"

# Prediction paths for all three models
PRED_PATHS = {
    '3-channel': "/path/to/3channel_prediction.png",   # Geometry-only (normal maps)
    '4-channel': "/path/to/4channel_prediction.png",   # Appearance-only (RGB + alpha)
    '7-channel': "/path/to/7channel_prediction.png"    # Combined (RGB + alpha + normals)
}

# Output directory for results
OUTPUT_DIR = "/path/to/output/"

# Stone Detection Parameters
# ---------------------------
# IoU threshold for considering a stone "detected"
# 0.5 = standard object detection threshold (COCO-style)
# 0.7 = stricter threshold requiring better boundary alignment
# 0.9 = very strict, only near-perfect matches
IOU_THRESHOLD = 0.5

# Minimum stone size (in pixels) to consider
# Helps filter out tiny noise artifacts
MIN_STONE_SIZE = 100  # pixels

# Class names (should match your color scheme)
CLASS_NAMES = ['Background', 'Ashlar', 'Polygonal', 'Quarry Stone']

# Model display colors for charts
MODEL_COLORS = {
    '3-channel': '#e74c3c',  # Red
    '4-channel': '#3498db',  # Blue
    '7-channel': '#2ecc71'   # Green
}

# Class colors for visualizations
CLASS_COLORS_DICT = {
    'Background': '#000000',
    'Ashlar': '#0000FF',
    'Polygonal': '#FF0000',
    'Quarry Stone': '#FFFF00'
}

print(f"Ground Truth: {GT_MASK_PATH}")
print(f"\nPrediction Paths:")
for model_name, path in PRED_PATHS.items():
    print(f"  {model_name}: {path}")
print(f"\nOutput Dir: {OUTPUT_DIR}")
print(f"\nDetection Parameters:")
print(f"  IoU Threshold: {IOU_THRESHOLD}")
print(f"  Min Stone Size: {MIN_STONE_SIZE} pixels")

---
## Cell 2: Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import seaborn as sns
from PIL import Image
import pandas as pd
from scipy import ndimage
from skimage.measure import label, regionprops
from typing import Dict, Tuple, List, Optional
from dataclasses import dataclass
import os
import warnings
warnings.filterwarnings('ignore')

Image.MAX_IMAGE_PIXELS = None

# RGB to class mapping
RGB_TO_CLASS = {
    (0, 0, 0): 0,       # Black -> Background
    (0, 0, 255): 1,     # Blue -> Ashlar
    (255, 0, 0): 2,     # Red -> Polygonal
    (255, 255, 0): 3    # Yellow -> Quarry Stone
}

CLASS_COLORS = ['#000000', '#0000FF', '#FF0000', '#FFFF00']

print("Imports complete.")

---
## Cell 3: Helper Functions

In [ ]:
@dataclass
class Stone:
    """Represents a single stone with its properties."""
    stone_id: int
    class_id: int
    class_name: str
    pixel_count: int
    mask: np.ndarray  # Boolean mask for this stone's pixels
    centroid: Tuple[float, float]
    bbox: Tuple[int, int, int, int]  # (min_row, min_col, max_row, max_col)


@dataclass
class StoneMatch:
    """Result of matching a GT stone to prediction."""
    gt_stone: Stone
    is_detected: bool
    iou: float
    predicted_class: Optional[int]
    is_correctly_classified: bool


def rgb_to_class_mask(rgb_image: np.ndarray, verbose: bool = True) -> np.ndarray:
    """
    Convert RGB mask to class indices.
    Handles unmapped colors by assigning to nearest class.
    """
    height, width = rgb_image.shape[:2]
    class_mask = np.zeros((height, width), dtype=np.uint8)
    
    for rgb_tuple, class_idx in RGB_TO_CLASS.items():
        color_mask = np.all(rgb_image == rgb_tuple, axis=2)
        class_mask[color_mask] = class_idx
    
    # Handle unmapped pixels (compression artifacts, anti-aliasing)
    mapped_pixels = np.zeros((height, width), dtype=bool)
    for rgb_tuple in RGB_TO_CLASS.keys():
        mapped_pixels |= np.all(rgb_image == rgb_tuple, axis=2)
    
    unmapped_count = np.sum(~mapped_pixels)
    if unmapped_count > 0 and verbose:
        print(f"    Note: {unmapped_count} pixels with unmapped colors -> mapped to nearest class")
        unmapped_indices = np.where(~mapped_pixels)
        for i in range(len(unmapped_indices[0])):
            y, x = unmapped_indices[0][i], unmapped_indices[1][i]
            pixel_rgb = rgb_image[y, x]
            min_dist = float('inf')
            nearest_class = 0
            for rgb_tuple, class_idx in RGB_TO_CLASS.items():
                dist = np.sqrt(np.sum((pixel_rgb.astype(float) - np.array(rgb_tuple).astype(float))**2))
                if dist < min_dist:
                    min_dist = dist
                    nearest_class = class_idx
            class_mask[y, x] = nearest_class
    
    return class_mask


def extract_stones_from_mask(
    class_mask: np.ndarray, 
    class_names: List[str],
    min_size: int = 100
) -> List[Stone]:
    """
    Extract individual stones from a class mask using connected component analysis.
    
    Each connected component of non-background pixels is considered one stone.
    The class of the stone is determined by majority vote of its pixels.
    
    Args:
        class_mask: 2D array with class indices (0=background, 1+=stone classes)
        class_names: List of class names
        min_size: Minimum pixel count to be considered a valid stone
        
    Returns:
        List of Stone objects
    """
    # Create binary mask of all stone pixels (non-background)
    stone_binary = (class_mask > 0).astype(np.uint8)
    
    # Label connected components
    labeled_array, num_features = ndimage.label(stone_binary)
    
    stones = []
    for region in regionprops(labeled_array):
        # Filter by size
        if region.area < min_size:
            continue
        
        # Create boolean mask for this stone
        stone_mask = (labeled_array == region.label)
        
        # Determine class by majority vote
        stone_classes = class_mask[stone_mask]
        class_id = int(np.bincount(stone_classes).argmax())
        
        stone = Stone(
            stone_id=region.label,
            class_id=class_id,
            class_name=class_names[class_id],
            pixel_count=region.area,
            mask=stone_mask,
            centroid=(region.centroid[0], region.centroid[1]),
            bbox=(region.bbox[0], region.bbox[1], region.bbox[2], region.bbox[3])
        )
        stones.append(stone)
    
    return stones


def calculate_iou(mask1: np.ndarray, mask2: np.ndarray) -> float:
    """Calculate IoU between two boolean masks."""
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    if union == 0:
        return 0.0
    return intersection / union


def match_stones_to_prediction(
    gt_stones: List[Stone],
    pred_mask: np.ndarray,
    iou_threshold: float = 0.5
) -> List[StoneMatch]:
    """
    Match each GT stone to the prediction and evaluate detection.
    
    For each GT stone:
    1. Find all predicted stone pixels that overlap with the GT stone
    2. Calculate IoU between GT stone and overlapping prediction region
    3. If IoU >= threshold, stone is considered detected
    4. Determine predicted class by majority vote of overlapping prediction pixels
    
    Args:
        gt_stones: List of Stone objects from ground truth
        pred_mask: Class prediction mask (same shape as GT)
        iou_threshold: Minimum IoU to consider a stone detected
        
    Returns:
        List of StoneMatch objects
    """
    matches = []
    
    for gt_stone in gt_stones:
        # Get prediction pixels that overlap with this GT stone
        pred_in_stone = pred_mask[gt_stone.mask]
        
        # Create prediction mask for stone region (non-background predictions)
        pred_stone_mask = (pred_mask > 0) & gt_stone.mask
        
        # Also check surrounding area - the prediction might extend beyond GT
        # Use the bounding box plus margin
        r_min, c_min, r_max, c_max = gt_stone.bbox
        margin = 20  # pixels
        r_min_ext = max(0, r_min - margin)
        c_min_ext = max(0, c_min - margin)
        r_max_ext = min(pred_mask.shape[0], r_max + margin)
        c_max_ext = min(pred_mask.shape[1], c_max + margin)
        
        # Create extended region mask
        extended_region = np.zeros_like(pred_mask, dtype=bool)
        extended_region[r_min_ext:r_max_ext, c_min_ext:c_max_ext] = True
        
        # Find predicted stones (connected components) that overlap with GT stone
        pred_in_region = (pred_mask > 0) & extended_region
        labeled_pred, _ = ndimage.label(pred_in_region)
        
        # Find which predicted components overlap with the GT stone
        overlapping_labels = np.unique(labeled_pred[gt_stone.mask])
        overlapping_labels = overlapping_labels[overlapping_labels > 0]  # Remove background
        
        if len(overlapping_labels) == 0:
            # No prediction overlaps with this GT stone
            match = StoneMatch(
                gt_stone=gt_stone,
                is_detected=False,
                iou=0.0,
                predicted_class=None,
                is_correctly_classified=False
            )
        else:
            # Find the best matching predicted component
            best_iou = 0.0
            best_pred_mask = None
            
            for pred_label in overlapping_labels:
                pred_component_mask = (labeled_pred == pred_label)
                iou = calculate_iou(gt_stone.mask, pred_component_mask)
                if iou > best_iou:
                    best_iou = iou
                    best_pred_mask = pred_component_mask
            
            is_detected = best_iou >= iou_threshold
            
            # Determine predicted class by majority vote
            if best_pred_mask is not None and best_pred_mask.sum() > 0:
                pred_classes = pred_mask[best_pred_mask]
                pred_classes = pred_classes[pred_classes > 0]  # Exclude background
                if len(pred_classes) > 0:
                    predicted_class = int(np.bincount(pred_classes).argmax())
                else:
                    predicted_class = None
            else:
                predicted_class = None
            
            is_correctly_classified = (
                is_detected and 
                predicted_class is not None and 
                predicted_class == gt_stone.class_id
            )
            
            match = StoneMatch(
                gt_stone=gt_stone,
                is_detected=is_detected,
                iou=best_iou,
                predicted_class=predicted_class,
                is_correctly_classified=is_correctly_classified
            )
        
        matches.append(match)
    
    return matches


def calculate_detection_metrics(
    matches: List[StoneMatch],
    class_names: List[str]
) -> Dict:
    """
    Calculate stone-level detection metrics from matches.
    
    Returns dict with:
    - total_stones: Number of GT stones
    - detected_stones: Number detected (IoU >= threshold)
    - detection_rate: detected / total
    - correctly_classified: Number with correct class
    - classification_accuracy: correctly_classified / detected
    - per_class: Dict with per-class breakdown
    - mean_iou_detected: Mean IoU of detected stones
    """
    total = len(matches)
    detected = sum(1 for m in matches if m.is_detected)
    correctly_classified = sum(1 for m in matches if m.is_correctly_classified)
    
    # Per-class breakdown (only stone classes, not background)
    per_class = {}
    for class_idx, class_name in enumerate(class_names):
        if class_idx == 0:  # Skip background
            continue
        class_matches = [m for m in matches if m.gt_stone.class_id == class_idx]
        class_total = len(class_matches)
        class_detected = sum(1 for m in class_matches if m.is_detected)
        class_correct = sum(1 for m in class_matches if m.is_correctly_classified)
        
        per_class[class_name] = {
            'total': class_total,
            'detected': class_detected,
            'detection_rate': class_detected / class_total if class_total > 0 else 0.0,
            'correctly_classified': class_correct,
            'classification_accuracy': class_correct / class_detected if class_detected > 0 else 0.0
        }
    
    # Mean IoU of detected stones
    detected_ious = [m.iou for m in matches if m.is_detected]
    mean_iou_detected = np.mean(detected_ious) if detected_ious else 0.0
    
    return {
        'total_stones': total,
        'detected_stones': detected,
        'detection_rate': detected / total if total > 0 else 0.0,
        'correctly_classified': correctly_classified,
        'classification_accuracy': correctly_classified / detected if detected > 0 else 0.0,
        'overall_accuracy': correctly_classified / total if total > 0 else 0.0,
        'per_class': per_class,
        'mean_iou_detected': mean_iou_detected
    }


print("Helper functions defined.")

---
## Cell 4: Load Ground Truth and Extract Stones

In [ ]:
# Load ground truth
print("Loading ground truth mask...")
gt_rgb = np.array(Image.open(GT_MASK_PATH).convert('RGB'))
gt_mask = rgb_to_class_mask(gt_rgb)
print(f"  Shape: {gt_mask.shape}")
print(f"  Classes present: {np.unique(gt_mask)}")

# Extract individual stones
print(f"\nExtracting individual stones (min size = {MIN_STONE_SIZE} px)...")
gt_stones = extract_stones_from_mask(gt_mask, CLASS_NAMES, MIN_STONE_SIZE)
print(f"  Total stones found: {len(gt_stones)}")

# Per-class stone counts
print("\n  Per-class stone counts:")
for class_idx, class_name in enumerate(CLASS_NAMES):
    if class_idx == 0:  # Skip background
        continue
    count = sum(1 for s in gt_stones if s.class_id == class_idx)
    print(f"    {class_name}: {count}")

# Stone size statistics
sizes = [s.pixel_count for s in gt_stones]
print(f"\n  Stone size statistics (pixels):")
print(f"    Min: {min(sizes):,}")
print(f"    Max: {max(sizes):,}")
print(f"    Mean: {np.mean(sizes):,.0f}")
print(f"    Median: {np.median(sizes):,.0f}")

print("\n✓ Ground truth stones extracted.")

---
## Cell 5: Visualize Extracted Stones

In [ ]:
# Create visualization showing individual stones
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original GT mask
cmap = ListedColormap(CLASS_COLORS)
axes[0].imshow(gt_mask, cmap=cmap, vmin=0, vmax=3)
axes[0].set_title(f'Ground Truth Mask\n({np.sum(gt_mask > 0):,} stone pixels)', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Individual stone labels (random colors)
stone_label_img = np.zeros(gt_mask.shape, dtype=np.int32)
for stone in gt_stones:
    stone_label_img[stone.mask] = stone.stone_id

# Use a colormap that shows each stone distinctly
np.random.seed(42)
n_stones = len(gt_stones)
random_colors = np.random.rand(n_stones + 1, 3)
random_colors[0] = [0, 0, 0]  # Background black
stone_cmap = ListedColormap(random_colors)

axes[1].imshow(stone_label_img, cmap=stone_cmap)
axes[1].set_title(f'Individual Stones\n({len(gt_stones)} stones detected)', fontsize=12, fontweight='bold')
axes[1].axis('off')

# Stone size distribution histogram
sizes = [s.pixel_count for s in gt_stones]
axes[2].hist(sizes, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[2].axvline(np.mean(sizes), color='red', linestyle='--', label=f'Mean: {np.mean(sizes):,.0f}')
axes[2].axvline(np.median(sizes), color='orange', linestyle='--', label=f'Median: {np.median(sizes):,.0f}')
axes[2].set_xlabel('Stone Size (pixels)', fontsize=11)
axes[2].set_ylabel('Count', fontsize=11)
axes[2].set_title('Stone Size Distribution', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Cell 6: Load All Predictions

In [ ]:
# Load all prediction masks
pred_masks = {}

print("Loading prediction masks...")
for model_name, pred_path in PRED_PATHS.items():
    print(f"\n  {model_name}:")
    pred_rgb = np.array(Image.open(pred_path).convert('RGB'))
    pred_mask = rgb_to_class_mask(pred_rgb, verbose=True)
    
    # Verify shape matches
    assert pred_mask.shape == gt_mask.shape, f"Shape mismatch for {model_name}: {pred_mask.shape} vs GT {gt_mask.shape}"
    
    pred_masks[model_name] = pred_mask
    print(f"    Shape: {pred_mask.shape}")
    print(f"    Classes predicted: {np.unique(pred_mask)}")

print("\n✓ All predictions loaded.")

---
## Cell 7: Match Stones and Calculate Detection Metrics

In [ ]:
# Match stones and calculate metrics for each model
all_results = {}
all_matches = {}

print(f"Matching stones (IoU threshold = {IOU_THRESHOLD})...\n")
print("="*70)

for model_name, pred_mask in pred_masks.items():
    print(f"\n{model_name.upper()}")
    print("-"*40)
    
    # Match GT stones to predictions
    matches = match_stones_to_prediction(gt_stones, pred_mask, IOU_THRESHOLD)
    all_matches[model_name] = matches
    
    # Calculate metrics
    metrics = calculate_detection_metrics(matches, CLASS_NAMES)
    all_results[model_name] = metrics
    
    # Print results
    print(f"  Total GT stones: {metrics['total_stones']}")
    print(f"  Detected stones: {metrics['detected_stones']} ({metrics['detection_rate']*100:.1f}%)")
    print(f"  Correctly classified: {metrics['correctly_classified']} ({metrics['overall_accuracy']*100:.1f}% of total)")
    print(f"  Classification accuracy (of detected): {metrics['classification_accuracy']*100:.1f}%")
    print(f"  Mean IoU (detected): {metrics['mean_iou_detected']:.4f}")
    
    print("\n  Per-class detection rates:")
    for class_name, class_metrics in metrics['per_class'].items():
        if class_metrics['total'] > 0:
            print(f"    {class_name:15s}: {class_metrics['detected']:3d}/{class_metrics['total']:3d} detected ({class_metrics['detection_rate']*100:5.1f}%), "
                  f"{class_metrics['correctly_classified']:3d} correct ({class_metrics['classification_accuracy']*100:5.1f}%)")

print("\n" + "="*70)
print("✓ All stone matching complete.")

---
## Cell 8: Comparative Summary Table

In [ ]:
# Build comparative summary table
summary_data = []

for model_name in PRED_PATHS.keys():
    metrics = all_results[model_name]
    row = {
        'Model': model_name,
        'Total_Stones': metrics['total_stones'],
        'Detected': metrics['detected_stones'],
        'Detection_Rate': metrics['detection_rate'],
        'Correctly_Classified': metrics['correctly_classified'],
        'Classification_Acc': metrics['classification_accuracy'],
        'Overall_Accuracy': metrics['overall_accuracy'],
        'Mean_IoU_Detected': metrics['mean_iou_detected']
    }
    
    # Add per-class detection rates
    for class_name in ['Ashlar', 'Polygonal', 'Quarry Stone']:
        if class_name in metrics['per_class']:
            row[f'{class_name}_Detection'] = metrics['per_class'][class_name]['detection_rate']
        else:
            row[f'{class_name}_Detection'] = np.nan
    
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)

# Display
print("\n" + "="*100)
print(f"STONE-LEVEL DETECTION SUMMARY (IoU Threshold = {IOU_THRESHOLD})")
print("="*100)

# Format percentages
display_df = summary_df.copy()
pct_cols = ['Detection_Rate', 'Classification_Acc', 'Overall_Accuracy', 
            'Ashlar_Detection', 'Polygonal_Detection', 'Quarry Stone_Detection']
for col in pct_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f'{x*100:.1f}%' if not np.isnan(x) else 'N/A')

display_df['Mean_IoU_Detected'] = display_df['Mean_IoU_Detected'].apply(lambda x: f'{x:.4f}')

print(display_df.to_string(index=False))
print("="*100)

# Highlight best model
best_model = summary_df.loc[summary_df['Detection_Rate'].idxmax(), 'Model']
best_rate = summary_df['Detection_Rate'].max()
print(f"\n→ Best detection rate: {best_model} ({best_rate*100:.1f}%)")

best_class_model = summary_df.loc[summary_df['Overall_Accuracy'].idxmax(), 'Model']
best_class_rate = summary_df['Overall_Accuracy'].max()
print(f"→ Best overall accuracy (detected + correct class): {best_class_model} ({best_class_rate*100:.1f}%)")

---
## Cell 9: Detection Rate Bar Chart

In [ ]:
# Bar chart comparing detection metrics across models
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_names = list(PRED_PATHS.keys())
colors = [MODEL_COLORS[m] for m in model_names]

# Detection Rate
detection_rates = [all_results[m]['detection_rate'] * 100 for m in model_names]
bars1 = axes[0].bar(model_names, detection_rates, color=colors, edgecolor='black', linewidth=1.5)
for bar, val in zip(bars1, detection_rates):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Detection Rate (%)', fontsize=12)
axes[0].set_title(f'Stone Detection Rate\n(IoU ≥ {IOU_THRESHOLD})', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 110)
axes[0].grid(axis='y', alpha=0.3)

# Classification Accuracy (of detected)
class_acc = [all_results[m]['classification_accuracy'] * 100 for m in model_names]
bars2 = axes[1].bar(model_names, class_acc, color=colors, edgecolor='black', linewidth=1.5)
for bar, val in zip(bars2, class_acc):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Classification Accuracy (%)', fontsize=12)
axes[1].set_title('Classification Accuracy\n(of detected stones)', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 110)
axes[1].grid(axis='y', alpha=0.3)

# Overall Accuracy (detected AND correctly classified)
overall_acc = [all_results[m]['overall_accuracy'] * 100 for m in model_names]
bars3 = axes[2].bar(model_names, overall_acc, color=colors, edgecolor='black', linewidth=1.5)
for bar, val in zip(bars3, overall_acc):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[2].set_ylabel('Overall Accuracy (%)', fontsize=12)
axes[2].set_title('Overall Accuracy\n(detected + correct class)', fontsize=14, fontweight='bold')
axes[2].set_ylim(0, 110)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Cell 10: Per-Class Detection Comparison

In [ ]:
# Grouped bar chart for per-class detection rates
fig, ax = plt.subplots(figsize=(12, 6))

stone_classes = ['Ashlar', 'Polygonal', 'Quarry Stone']
x = np.arange(len(stone_classes))
width = 0.25
multiplier = 0

for model_name in PRED_PATHS.keys():
    detection_rates = []
    for class_name in stone_classes:
        if class_name in all_results[model_name]['per_class']:
            rate = all_results[model_name]['per_class'][class_name]['detection_rate'] * 100
        else:
            rate = 0
        detection_rates.append(rate)
    
    offset = width * multiplier
    bars = ax.bar(x + offset, detection_rates, width, 
                  label=model_name, color=MODEL_COLORS[model_name],
                  edgecolor='black', linewidth=1)
    
    # Add value labels
    for bar, val in zip(bars, detection_rates):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    multiplier += 1

ax.set_ylabel('Detection Rate (%)', fontsize=12)
ax.set_title(f'Per-Class Stone Detection Rate (IoU ≥ {IOU_THRESHOLD})', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(stone_classes, fontsize=11)
ax.legend(loc='upper right', fontsize=11)
ax.set_ylim(0, 115)
ax.grid(axis='y', alpha=0.3)

# Add stone counts below class names
for i, class_name in enumerate(stone_classes):
    # Get total stones for this class from any model result (they're all the same)
    first_model = list(PRED_PATHS.keys())[0]
    if class_name in all_results[first_model]['per_class']:
        total = all_results[first_model]['per_class'][class_name]['total']
        ax.text(i + width, -8, f'(n={total})', ha='center', fontsize=10, color='gray')

plt.tight_layout()
plt.show()

---
## Cell 11: IoU Distribution of Detected Stones

In [ ]:
# IoU distribution histograms for each model
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, model_name in enumerate(PRED_PATHS.keys()):
    matches = all_matches[model_name]
    ious = [m.iou for m in matches]
    
    # Split by detected vs not detected
    detected_ious = [m.iou for m in matches if m.is_detected]
    undetected_ious = [m.iou for m in matches if not m.is_detected]
    
    axes[idx].hist(detected_ious, bins=20, alpha=0.7, color=MODEL_COLORS[model_name], 
                   label=f'Detected (n={len(detected_ious)})', edgecolor='black')
    axes[idx].hist(undetected_ious, bins=20, alpha=0.5, color='gray',
                   label=f'Not detected (n={len(undetected_ious)})', edgecolor='black')
    
    # Mark threshold
    axes[idx].axvline(IOU_THRESHOLD, color='red', linestyle='--', linewidth=2, 
                      label=f'Threshold ({IOU_THRESHOLD})')
    
    axes[idx].set_xlabel('IoU with Ground Truth', fontsize=11)
    axes[idx].set_ylabel('Number of Stones', fontsize=11)
    axes[idx].set_title(f'{model_name}\nMean IoU (detected): {np.mean(detected_ious):.3f}' if detected_ious else model_name,
                        fontsize=12, fontweight='bold')
    axes[idx].legend(fontsize=9)
    axes[idx].set_xlim(0, 1)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Cell 12: Visualize Detection Results

In [ ]:
# Create detection visualization for each model
# Green = correctly detected and classified
# Yellow = detected but wrong class
# Red = not detected

fig, axes = plt.subplots(2, 2, figsize=(16, 16))

# Ground truth
cmap = ListedColormap(CLASS_COLORS)
axes[0, 0].imshow(gt_mask, cmap=cmap, vmin=0, vmax=3)
axes[0, 0].set_title(f'Ground Truth\n({len(gt_stones)} stones)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Detection results for each model
model_list = list(PRED_PATHS.keys())
positions = [(0, 1), (1, 0), (1, 1)]

for (row, col), model_name in zip(positions, model_list):
    matches = all_matches[model_name]
    metrics = all_results[model_name]
    
    # Create RGB visualization
    vis_img = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    vis_img[gt_mask == 0] = [30, 30, 30]  # Dark gray background
    
    for match in matches:
        if match.is_correctly_classified:
            color = [0, 200, 0]  # Green - correctly detected and classified
        elif match.is_detected:
            color = [255, 200, 0]  # Yellow - detected but wrong class
        else:
            color = [200, 0, 0]  # Red - not detected
        
        vis_img[match.gt_stone.mask] = color
    
    axes[row, col].imshow(vis_img)
    axes[row, col].set_title(
        f'{model_name}\n'
        f'Detection: {metrics["detection_rate"]*100:.1f}% | '
        f'Overall Acc: {metrics["overall_accuracy"]*100:.1f}%',
        fontsize=12, fontweight='bold'
    )
    axes[row, col].axis('off')

# Add legend
legend_elements = [
    Patch(facecolor='#00C800', label='Correct (detected + right class)'),
    Patch(facecolor='#FFC800', label='Detected but wrong class'),
    Patch(facecolor='#C80000', label='Not detected')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11, 
           bbox_to_anchor=(0.5, 0.02))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

---
## Cell 13: Threshold Sensitivity Analysis

In [ ]:
# Analyze how detection rate changes with different IoU thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name in PRED_PATHS.keys():
    matches = all_matches[model_name]
    
    detection_rates = []
    overall_accs = []
    
    for thresh in thresholds:
        detected = sum(1 for m in matches if m.iou >= thresh)
        correct = sum(1 for m in matches if m.iou >= thresh and 
                      m.predicted_class == m.gt_stone.class_id)
        
        detection_rates.append(detected / len(matches) * 100)
        overall_accs.append(correct / len(matches) * 100)
    
    axes[0].plot(thresholds, detection_rates, 'o-', color=MODEL_COLORS[model_name], 
                 label=model_name, linewidth=2, markersize=8)
    axes[1].plot(thresholds, overall_accs, 'o-', color=MODEL_COLORS[model_name],
                 label=model_name, linewidth=2, markersize=8)

# Mark current threshold
for ax in axes:
    ax.axvline(IOU_THRESHOLD, color='gray', linestyle='--', alpha=0.7, 
               label=f'Current threshold ({IOU_THRESHOLD})')

axes[0].set_xlabel('IoU Threshold', fontsize=12)
axes[0].set_ylabel('Detection Rate (%)', fontsize=12)
axes[0].set_title('Detection Rate vs IoU Threshold', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 105)

axes[1].set_xlabel('IoU Threshold', fontsize=12)
axes[1].set_ylabel('Overall Accuracy (%)', fontsize=12)
axes[1].set_title('Overall Accuracy vs IoU Threshold', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

---
## Cell 14: Save Results

In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save summary table
summary_path = os.path.join(OUTPUT_DIR, 'stone_detection_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f"✓ Summary table saved to: {summary_path}")

# Save detailed per-stone results for each model
for model_name in PRED_PATHS.keys():
    matches = all_matches[model_name]
    
    stone_data = []
    for match in matches:
        stone_data.append({
            'stone_id': match.gt_stone.stone_id,
            'gt_class': match.gt_stone.class_name,
            'pixel_count': match.gt_stone.pixel_count,
            'centroid_row': match.gt_stone.centroid[0],
            'centroid_col': match.gt_stone.centroid[1],
            'is_detected': match.is_detected,
            'iou': match.iou,
            'predicted_class': CLASS_NAMES[match.predicted_class] if match.predicted_class is not None else 'None',
            'is_correctly_classified': match.is_correctly_classified
        })
    
    stone_df = pd.DataFrame(stone_data)
    stone_path = os.path.join(OUTPUT_DIR, f'stone_detection_{model_name.replace("-", "_")}.csv')
    stone_df.to_csv(stone_path, index=False)
    print(f"✓ Per-stone results for {model_name} saved to: {stone_path}")

# Save threshold sensitivity data
threshold_data = []
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    row = {'threshold': thresh}
    for model_name in PRED_PATHS.keys():
        matches = all_matches[model_name]
        detected = sum(1 for m in matches if m.iou >= thresh)
        correct = sum(1 for m in matches if m.iou >= thresh and 
                      m.predicted_class == m.gt_stone.class_id)
        row[f'{model_name}_detection_rate'] = detected / len(matches)
        row[f'{model_name}_overall_acc'] = correct / len(matches)
    threshold_data.append(row)

threshold_df = pd.DataFrame(threshold_data)
threshold_path = os.path.join(OUTPUT_DIR, 'threshold_sensitivity.csv')
threshold_df.to_csv(threshold_path, index=False)
print(f"✓ Threshold sensitivity data saved to: {threshold_path}")

print("\n✓ All results saved.")

---
## Cell 15: Summary Statistics for Paper

In [ ]:
# Generate summary text suitable for paper
print("="*70)
print("SUMMARY FOR PAPER")
print("="*70)

print(f"\nDataset: {len(gt_stones)} individual stones across {len([c for c in np.unique(gt_mask) if c > 0])} masonry classes")
print(f"IoU Threshold: {IOU_THRESHOLD}")
print(f"Minimum stone size: {MIN_STONE_SIZE} pixels\n")

print("Per-class stone counts:")
for class_name in ['Ashlar', 'Polygonal', 'Quarry Stone']:
    count = sum(1 for s in gt_stones if s.class_name == class_name)
    print(f"  {class_name}: {count}")

print("\n" + "-"*70)
print("Model Comparison (Stone-Level Metrics):")
print("-"*70)

for model_name in PRED_PATHS.keys():
    metrics = all_results[model_name]
    print(f"\n{model_name}:")
    print(f"  Stone Detection Rate: {metrics['detection_rate']*100:.1f}%")
    print(f"  Classification Accuracy (of detected): {metrics['classification_accuracy']*100:.1f}%")
    print(f"  Overall Accuracy (detected + correct): {metrics['overall_accuracy']*100:.1f}%")
    print(f"  Mean IoU of detected stones: {metrics['mean_iou_detected']:.3f}")

print("\n" + "="*70)